In [1]:
%run ../setup.ipynb

In [2]:
tablename = "stadium"
url = "https://leetcode.com/problems/human-traffic-of-stadium/description/"

In [3]:
print(f"{url}")
print(description_from_url(url))

https://leetcode.com/problems/human-traffic-of-stadium/description/
Table: Stadium

+---------------+---------+
| Column Name   | Type    |
+---------------+---------+
| id            | int     |
| visit_date    | date    |
| people        | int     |
+---------------+---------+
visit_date is the column with unique values for this table.
Each row of this table contains the visit date and visit id to the stadium with the number of people during the visit.
As the id increases, the date increases as well.

 
Write a solution to display the records with three or more rows with consecutive id's, and the number of people is greater than or equal to 100 for each.
Return the result table ordered by visit_date in ascending order.
The result format is in the following example.
 
Example 1:

Input: 
Stadium table:
+------+------------+-----------+
| id   | visit_date | people    |
+------+------------+-----------+
| 1    | 2017-01-01 | 10        |
| 2    | 2017-01-02 | 109       |
| 3    | 2017-0

In [4]:
convert_ascii_to_csv(f"{tablename}.csv")

Completed writing data to stadium_converted.csv


In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql import functions as f

In [6]:
spark = SparkSession.builder.appName("Human traffic of stadium").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/16 00:23:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
df = spark.read.option("header","true")\
.format("csv")\
.load(f"{tablename}_converted.csv")

In [8]:
df.show()

+---+----------+------+
| id|visit_date|people|
+---+----------+------+
|  1|2017-01-01|    10|
|  2|2017-01-02|   109|
|  3|2017-01-03|   150|
|  4|2017-01-04|    99|
|  5|2017-01-05|   145|
|  6|2017-01-06|  1455|
|  7|2017-01-07|   199|
|  8|2017-01-09|   188|
+---+----------+------+



In [9]:
df.createOrReplaceTempView(tablename)

In [10]:
ws = Window.orderBy(f.col("id"))

In [11]:
df.filter(f.col("people")>=100)\
    .withColumn("rownum", f.row_number().over(ws))\
    .withColumn("grp", f.col("id")-f.col("rownum"))\
    .withColumn("consecutive_num", f.count("id").over(Window.partitionBy("grp")))\
    .filter(f.col("consecutive_num")>=3)\
    .select("id", "visit_date", "people")\
    .orderBy(f.col("id"))\
    .show()

26/02/16 00:24:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/16 00:24:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/16 00:24:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/16 00:24:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---+----------+------+
| id|visit_date|people|
+---+----------+------+
|  5|2017-01-05|   145|
|  6|2017-01-06|  1455|
|  7|2017-01-07|   199|
|  8|2017-01-09|   188|
+---+----------+------+



In [12]:
spark.sql(
    """
    with grouped_cte as
    (
    select id, visit_date, people, 
    (id - row_number() over(order by id)) as grp from stadium where people>=100
    ),
    consec_num as
    (
    select id, visit_date, people,
    count(id) over(partition by grp) as consec_num from grouped_cte
    )
    select id, visit_date, people from consec_num where consec_num>=3
    """
).show()

26/02/16 00:24:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/16 00:24:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/16 00:24:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/16 00:24:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---+----------+------+
| id|visit_date|people|
+---+----------+------+
|  5|2017-01-05|   145|
|  6|2017-01-06|  1455|
|  7|2017-01-07|   199|
|  8|2017-01-09|   188|
+---+----------+------+

